### RAG Pipelines- Data Ingestion to Vector DB Pipeline


In [19]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [20]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: Interview Prep Playbook GenAI Agentic AI Developer.pdf
  ✓ Loaded 12 pages

Processing: Recruitment of Managers and Junior Executives in Various Disciplines.pdf
  ✓ Loaded 3 pages

Total documents loaded: 15


In [21]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m156', 'creator': 'Chromium', 'creationdate': '2026-11-09T20:59:21+00:00', 'title': 'Interview Prep Playbook: GenAI / Agentic AI Developer', 'moddate': '2026-11-09T20:59:21+00:00', 'source': '..\\data\\pdf_files\\Interview Prep Playbook GenAI Agentic AI Developer.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'Interview Prep Playbook GenAI Agentic AI Developer.pdf', 'file_type': 'pdf'}, page_content='Interview Prep Playbook: GenAI / Agentic AI\nDeveloper\nPrepared for Siddarth Singh · 7-day roadmap and cheat sheet · Built from your resume and\nthe job description you shared\nHow to use this document\nRead Section 1 today so you know where you stand. Drill Section 2 out loud, rehearse the\nSection 3 scripts until they sound like you, follow Section 4 day by day, and keep Section 5\nfor the end of the interview. Anything in [square brackets] is a fact only you can supply.\nNever invent numbers or claim experience you don’t 

In [22]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [23]:
chunks=split_documents(all_pdf_documents)
chunks

Split 15 documents into 53 chunks

Example chunk:
Content: Interview Prep Playbook: GenAI / Agentic AI
Developer
Prepared for Siddarth Singh · 7-day roadmap and cheat sheet · Built from your resume and
the job description you shared
How to use this document
R...
Metadata: {'producer': 'Skia/PDF m156', 'creator': 'Chromium', 'creationdate': '2026-11-09T20:59:21+00:00', 'title': 'Interview Prep Playbook: GenAI / Agentic AI Developer', 'moddate': '2026-11-09T20:59:21+00:00', 'source': '..\\data\\pdf_files\\Interview Prep Playbook GenAI Agentic AI Developer.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'Interview Prep Playbook GenAI Agentic AI Developer.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m156', 'creator': 'Chromium', 'creationdate': '2026-11-09T20:59:21+00:00', 'title': 'Interview Prep Playbook: GenAI / Agentic AI Developer', 'moddate': '2026-11-09T20:59:21+00:00', 'source': '..\\data\\pdf_files\\Interview Prep Playbook GenAI Agentic AI Developer.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'Interview Prep Playbook GenAI Agentic AI Developer.pdf', 'file_type': 'pdf'}, page_content='Interview Prep Playbook: GenAI / Agentic AI\nDeveloper\nPrepared for Siddarth Singh · 7-day roadmap and cheat sheet · Built from your resume and\nthe job description you shared\nHow to use this document\nRead Section 1 today so you know where you stand. Drill Section 2 out loud, rehearse the\nSection 3 scripts until they sound like you, follow Section 4 day by day, and keep Section 5\nfor the end of the interview. Anything in [square brackets] is a fact only you can supply.\nNever invent numbers or claim experience you don’t 

### embedding And vectorStoreDB

In [24]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [25]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1484.38it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\sidda\AppData\Local\Temp\ipykernel_2628\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### Vector Store

In [26]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents_cosine", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG", "hnsw:space": "cosine"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents_cosine
Existing documents in collection: 53


In [27]:
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 53 texts...


Batches: 100%|██████████| 2/2 [00:04<00:00,  2.11s/it]


Generated embeddings with shape: (53, 384)
Adding 53 documents to vector store...
Successfully added 53 documents to vector store
Total documents in collection: 106


In [28]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [29]:
rag_retriever 

In [30]:
rag_retriever.retrieve("What is the Junior Executive")

Retrieving documents for query: 'What is the Junior Executive'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 24.07it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_37004577_8',
  'content': 'rerank → grounded prompt with citations → evaluate (recall@k, groundedness).\nQ3. “What’s the difference between a chain and an agent? How does tool/function\ncalling work, and what do you do when the model calls a tool wrongly?”\nJD trigger: “Develop AI agents, agentic workflows… LangChain, LangGraph, AutoGen,\nCrewAI, Semantic Kernel, Amazon Bedrock Agents”; “tool/function calling”.\nTesting: your mental model of the agent loop and how you handle its failure modes.\nYour angle: fixed sequence versus model-driven loop; JSON schemas; validate\narguments before executing; cap iterations; retry with feedback; log every step.\nInterview Prep Playbook: GenAI / Agentic AI Developer\nPage 3 of 12',
  'metadata': {'source_file': 'Interview Prep Playbook GenAI Agentic AI Developer.pdf',
   'total_pages': 12,
   'content_length': 690,
   'title': 'Interview Prep Playbook: GenAI / Agentic AI Developer',
   'page': 2,
   'creator': 'Chromium',
   'source': 

In [31]:
rag_retriever.retrieve("Your 3 biggest risk areas")

Retrieving documents for query: 'Your 3 biggest risk areas'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 39.38it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_16c85b13_4',
  'content': 'signature line.\n2. Enterprise integration experience in client-facing, regulated settings. US Bank and\nan insurance/care-management client, with Lambda, Amazon Connect, webhooks,\nthird-party APIs and secure data storage. This maps straight onto “integrations with\nLLMs, APIs, enterprise applications, databases”.\n3. Direct exposure to the vendors the JD names. Bedrock and Anthropic are both listed\nas preferred, and you hold AWS AI Practitioner and (in progress or complete) the\nAnthropic Claude Foundation Developer certification, plus a Gemini API build. That\nshows initiative beyond your day job.\nYour 3 biggest risk areas\n1. Experience level. The JD says 2–5 years and “production-quality applications”.\nExpect probing on testing, debugging and performance. Fix: quantify your work, add\nthe missing US Bank dates, and be ready to show a Python repo (see the Day 2–4\nbuild).\n2. The agentic and RAG gap on paper. This is the heart of the role,

In [32]:
rag_retriever.retrieve("Your 3 biggest risk areas")


Retrieving documents for query: 'Your 3 biggest risk areas'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 15.44it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_16c85b13_4',
  'content': 'signature line.\n2. Enterprise integration experience in client-facing, regulated settings. US Bank and\nan insurance/care-management client, with Lambda, Amazon Connect, webhooks,\nthird-party APIs and secure data storage. This maps straight onto “integrations with\nLLMs, APIs, enterprise applications, databases”.\n3. Direct exposure to the vendors the JD names. Bedrock and Anthropic are both listed\nas preferred, and you hold AWS AI Practitioner and (in progress or complete) the\nAnthropic Claude Foundation Developer certification, plus a Gemini API build. That\nshows initiative beyond your day job.\nYour 3 biggest risk areas\n1. Experience level. The JD says 2–5 years and “production-quality applications”.\nExpect probing on testing, debugging and performance. Fix: quantify your work, add\nthe missing US Bank dates, and be ready to show a Python repo (see the Day 2–4\nbuild).\n2. The agentic and RAG gap on paper. This is the heart of the role,

In [33]:
rag_retriever.retrieve("Your 3 biggest risk areas")

Retrieving documents for query: 'Your 3 biggest risk areas'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 28.56it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_16c85b13_4',
  'content': 'signature line.\n2. Enterprise integration experience in client-facing, regulated settings. US Bank and\nan insurance/care-management client, with Lambda, Amazon Connect, webhooks,\nthird-party APIs and secure data storage. This maps straight onto “integrations with\nLLMs, APIs, enterprise applications, databases”.\n3. Direct exposure to the vendors the JD names. Bedrock and Anthropic are both listed\nas preferred, and you hold AWS AI Practitioner and (in progress or complete) the\nAnthropic Claude Foundation Developer certification, plus a Gemini API build. That\nshows initiative beyond your day job.\nYour 3 biggest risk areas\n1. Experience level. The JD says 2–5 years and “production-quality applications”.\nExpect probing on testing, debugging and performance. Fix: quantify your work, add\nthe missing US Bank dates, and be ready to show a Python repo (see the Day 2–4\nbuild).\n2. The agentic and RAG gap on paper. This is the heart of the role,

In [34]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

groq_api_key = os.getenv("GROQ_API_KEY")
llm=ChatGroq(api_key=groq_api_key, model="gemma2-9b-it", temperature=0.1, max_tokens=500)
def rag_simple(query, retriever,llm,top_k=3):
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant documents found for the query."

    promt="Any information you need is in the context below. Use it to answer the question. If you cannot find the answer, say 'I don't know'.\n\nContext:\n" + context + "\n\nQuestion: " + query
    response=llm.invoke([prompt.format(context=context, query=query)])
    return response.content


